In [ ]:
# !pip3 install bibtexparser

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.


In [10]:
import bibtexparser
from bibtexparser.bparser import BibTexParser
from bibtexparser.customization import homogenize_latex_encoding
import re

def normalize_title(title):
    title = homogenize_latex_encoding({"title": title})["title"]
    """Normalize title for comparison: lowercase, remove spaces and punctuation."""
    return re.sub(r'\W+', '', title.lower().strip())

def load_bibtex_file(filepath):
    with open(filepath, 'r', encoding='utf-8') as bibfile:
        parser = BibTexParser(common_strings=True)
        # parser.customization = homogenize_latex_encoding
        return bibtexparser.load(bibfile, parser=parser)

# Load both BibTeX files
detailed = load_bibtex_file("detailed-bibtex.bib")
fallback = load_bibtex_file("google-scholar-export.bib")

# Index entries by normalized title
merged_entries = {}
for entry in fallback.entries:
    title = entry.get("title", "")
    if not title or not entry.get("year"):
        continue
    norm_title = normalize_title(title)
    merged_entries[norm_title] = entry

# Prefer detailed entries if they exist and have a year
for entry in detailed.entries:
    title = entry.get("title", "")
    if not title or not entry.get("year"):
        continue
    norm_title = normalize_title(title)
    merged_entries[norm_title] = entry

# Create output bib database
output_db = bibtexparser.bibdatabase.BibDatabase()
output_db.entries = list(merged_entries.values())

# Dump to papers.bib
writer = bibtexparser.bwriter.BibTexWriter()
with open("papers-out.bib", "w", encoding="utf-8") as outfile:
    outfile.write(writer.write(output_db))

print(f"✔ Merged {len(output_db.entries)} entries into papers.bib")


✔ Merged 333 entries into papers.bib
